# Testando e comparando os modelos de Detecção de Emoções

## Etapa 1 - Importando as bibliotecas

In [ ]:
# bibliotecas básicas que vamos usar no notebook
import cv2
import numpy as np
import pandas as pd
from google.colab.patches import cv2_imshow
import zipfile
%tensorflow_version 2.x
import tensorflow
tensorflow.__version__

## Etapa 2 - Conectando com o Drive e acessando os arquivos

In [ ]:
# conecta com o drive e extrai o material do curso, que já contém os modelos treinados (.h5)
from google.colab import drive
drive.mount('/content/gdrive')
path = "/content/gdrive/My Drive/Material.zip"
zip_object = zipfile.ZipFile(file=path, mode="r")
zip_object.extractall("./")
zip_object.close

# Faz uma comparação

In [ ]:
from tensorflow.keras.models import load_model
import operator

# nome dos modelos que serão comparados (arquivos .h5 salvos nos notebooks de cada arquitetura)
arquivos_modelos = ["modelo_01_expressoes.h5", "modelo_02_expressoes.h5", "modelo_03_expressoes.h5", "modelo_04_expressoes.h5", "modelo_05_expressoes.h5"]

modelos = {} # vai guardar a acurácia de cada modelo, pra depois ordenar

# carrega o mesmo conjunto de teste usado no treinamento de cada modelo (pra comparação ser justa)
x_test = np.load('Material/mod_xtest.npy')
y_test = np.load('Material/mod_ytest.npy')

for modelo in arquivos_modelos:
  model = load_model('Material/' + modelo)

  # calcula a acurácia do modelo obtida ao testar na base de teste
  scores = model.evaluate(np.array(x_test), np.array(y_test), batch_size=64)
  print("---"+ str(modelo) +"---")
  print("Perda/Loss: " + str(scores[0]))
  print("Acurácia: " + str(scores[1]))
  modelos[modelo] = str(scores[1])
  print("\n")

In [ ]:
# ordena em ordem decrescente os modelos com base no valor da acurácia
order_modelos = sorted(modelos.items(), key=operator.itemgetter(1), reverse=True)
print(order_modelos)

In [ ]:
order_modelos[0] # o primeiro da lista é o modelo com maior acurácia

## Teste com o modelo carregado

In [ ]:
# carrega uma foto de teste pra testar o modelo vencedor
imagem = cv2.imread("Material/testes/teste_gabriel.png")
cv2_imshow(imagem)

In [ ]:
cascade_faces = 'Material/haarcascade_frontalface_default.xml'
caminho_modelo = 'Material/' + str(order_modelos[0][0])  # pega o nome do modelo que teve a maior acurácia
face_detection = cv2.CascadeClassifier(cascade_faces)
classificador_emocoes = load_model(caminho_modelo, compile=False)
expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

In [ ]:
from tensorflow.keras.models import load_model
from tensorflow.keras.preprocessing.image import img_to_array

# carrega o modelo vencedor
face_detection = cv2.CascadeClassifier(cascade_faces)
classificador_emocoes = load_model(caminho_modelo, compile=False)

expressoes = ["Raiva", "Nojo", "Medo", "Feliz", "Triste", "Surpreso", "Neutro"]

original = imagem.copy()
faces = face_detection.detectMultiScale(original,scaleFactor=1.1,minNeighbors=3,minSize=(20,20))
cinza = cv2.cvtColor(original, cv2.COLOR_BGR2GRAY)

if len(faces) > 0:
    for (fX, fY, fW, fH) in faces:
      roi = cinza[fY:fY + fH, fX:fX + fW] # extrai só a região do rosto (ROI)
      roi = cv2.resize(roi, (48, 48)) # redimensiona pro tamanho esperado pelo modelo
      roi = roi.astype("float") / 255.0 # normaliza
      roi = img_to_array(roi) # converte pra array que a rede consegue processar
      roi = np.expand_dims(roi, axis=0)
      preds = classificador_emocoes.predict(roi)[0] # prediz a probabilidade de cada emoção
      print(preds)
      emotion_probability = np.max(preds)
      label = expressoes[preds.argmax()]# pega a emoção com maior probabilidade
      cv2.putText(original, label, (fX, fY - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.65, (0, 0, 255), 2, cv2.LINE_AA)
      cv2.rectangle(original, (fX, fY), (fX + fW, fY + fH),(0, 0, 255), 2)
else:
    print('Nenhuma face detectada')


cv2_imshow(original)

# desenha as barras com a probabilidade de cada emoção (só faz sentido mostrar se for 1 rosto só)
probabilidades = np.ones((250, 300, 3), dtype="uint8") * 255
if len(faces) == 1:
  for (i, (emotion, prob)) in enumerate(zip(expressoes, preds)):
      # nome das emoções
      text = "{}: {:.2f}%".format(emotion, prob * 100)
      w = int(prob * 300) # tamanho da barra proporcional à probabilidade
      cv2.rectangle(probabilidades, (7, (i * 35) + 5),
      (w, (i * 35) + 35), (200, 250, 20), -1)
      cv2.putText(probabilidades, text, (10, (i * 35) + 23),
      cv2.FONT_HERSHEY_SIMPLEX, 0.45,
      (0, 0, 0), 1, cv2.LINE_AA)

  cv2_imshow(probabilidades)

cv2.imwrite("captura.jpg",original) # salva o resultado como imagem
cv2.destroyAllWindows()